# 01 - Load the dataset

**What this does:** loads `data/processed/posts.parquet` — the ONE merged
dataset (7.95M posts, 15 subreddits, 2008-2025) — and lets you slice it by
subreddit and date without touching the raw files.

**This notebook no longer reads `.zst` files.** The raw-to-parquet conversion
already happened (it's slow — ~3 GB of compressed JSON) and lives in
`data_ingestion/scripts/prep_posts.py`. You only re-run that script if the
raw dumps change or you want different cleaning rules. Day-to-day analysis
starts HERE, from the parquet, which loads in seconds.

Because the parquet is *columnar* and stored in subreddit blocks, we can read
just the columns and subreddits we need instead of the whole 1.15 GB file.

In [32]:
import os, sys
# Find the project root so we can import the src/ package and locate data/.
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
print('project root:', ROOT)

project root: c:\Users\alexd\Desktop\GIC\RetailFlow1


In [33]:
# ============ TIME WINDOW - edit freely ============
# Keep this narrow to keep runs fast. Rough guide with ALL 15 subreddits:
#   1 year  ~ 700k posts  -> notebook 02 scan: a few minutes
#   all dates = 7.9M posts -> 02: ~30-60 min, 04 (themes): many HOURS
# Set both to None for the full 2008-2025 history.
START_DATE = '2021-01-01'   # inclusive,  'YYYY-MM-DD' or None
END_DATE   = '2022-01-01'   # EXCLUSIVE,  'YYYY-MM-DD' or None
# ====================================================

In [34]:
# ============ PARAMETERS - edit these ============
POSTS_PATH = os.path.join(ROOT, 'data', 'processed', 'posts.parquet')
SUBREDDITS = []      # e.g. ['wallstreetbets', 'stocks'];  [] = ALL 15 subreddits
COLUMNS    = None    # e.g. ['date','subreddit','title','score'];  None = all 8
SLICE_OUT  = None    # e.g. os.path.join(ROOT,'data','processed','posts_slice.parquet')
                     # to save the filtered slice for the other notebooks, or None
# ==================================================

In [35]:
# Quick metadata peek - reads only the file FOOTER, so it is instant
# even though the file is 1.15 GB.
import pyarrow.parquet as pq

pf = pq.ParquetFile(POSTS_PATH)
print('rows       :', f'{pf.metadata.num_rows:,}')
print('columns    :', pf.schema_arrow.names)
print('row groups :', pf.metadata.num_row_groups)

rows       : 7,954,297
columns    : ['id', 'date', 'author', 'score', 'subreddit', 'title', 'selftext', 'num_comments']
row groups : 34


In [36]:
# Load the data - with filters pushed down into the parquet reader.
# pyarrow uses each row group's min/max stats to SKIP blocks that can't
# match, so asking for one subreddit reads ~1/30th of the file.

### IMPT: This takes a while but works

import pandas as pd

filters = []
if SUBREDDITS:
    filters.append(('subreddit', 'in', [s.lower() for s in SUBREDDITS]))
if START_DATE:
    filters.append(('date', '>=', START_DATE))
if END_DATE:
    filters.append(('date', '<', END_DATE))

posts = pq.read_table(
    POSTS_PATH,
    columns=COLUMNS,
    filters=filters if filters else None,
).to_pandas()

print('loaded', f'{len(posts):,}', 'posts')
print()
print(posts['subreddit'].value_counts())

loaded 2,832,937 posts

subreddit
wallstreetbets           1407544
cryptocurrency            680919
bitcoin                   166716
personalfinance           154782
stocks                    103950
pennystocks                81807
investing                  64401
stockmarket                56751
options                    37489
daytrading                 29716
thetagang                  16647
financialindependence      13079
dividends                  10788
valueinvesting              6733
securityanalysis            1615
Name: count, dtype: int64


In [37]:
# Preview.
posts.head()

,id,date,author,score,subreddit,title,selftext,num_comments
0,ko10pg,2021-01-01,[deleted],0,bitcoin,First Time Saved/ Made Money,[deleted],6
1,ko12mk,2021-01-01,randum-guy,0,bitcoin,Btc dip to 20k?,Is it possible for Bitcoin to dip to 20k? My b...,19
2,ko15jt,2021-01-01,Mari0805,119,bitcoin,BTC just had the monthly and yearly close! 202...,Let's see what 2021 brings us. I predict 2021 ...,29
3,ko171i,2021-01-01,[deleted],0,bitcoin,I believe in Bitcoin.,[deleted],5
4,ko1877,2021-01-01,[deleted],1,bitcoin,Please help me find a solution with my BTC wal...,[deleted],28


In [38]:
# Optionally save the slice so notebooks 02/04 can point POSTS_PATH at it.
if SLICE_OUT:
    posts.to_parquet(SLICE_OUT, index=False)
    print('saved', f'{len(posts):,}', 'posts ->', SLICE_OUT)
else:
    print('SLICE_OUT is None - nothing saved (set it in the PARAMETERS cell to export a slice)')

SLICE_OUT is None - nothing saved (set it in the PARAMETERS cell to export a slice)
